In [ ]:
import pandas as pd
import os
from datetime import datetime
import re

In [ ]:
## 주의사항 ##

# 이 이전에 이미 Doc_ID, Pub_Date, Col_Date, Title, Text, Filename 컬럼이 있는 데이터프레임이 만들어져 있어야함 ##
## Doc_ID는 맘대로 만든거고, Sen_ID를 만들 때 사용되는거라 Sen_ID부분 지우고 하면 Doc_ID 안만들어도 될 듯 함 ##
## 토큰화 할 때 Text 컬럼으로 만드는 거라서 나머지 컬럼들은 꼭 없어도 되긴 함 ##
## 그럴 경우 tokenizing 함수에서 맨 마지막 'Doc_ID', 'Filename', 'Title', 'Pub_Type', 'Pub_Subj', 'Pub_Date', 'Col_Date', 'Sen_ID', 이 컬럼들을 지우고 해야할듯 ##

<br>
<br>
<br>

# 텍스트 정제

In [ ]:
import re

def clean_text(text):
    # 이모티콘 제거
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F700-\U0001F77F"  # alchemical symbols
                               u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                               u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                               u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                               u"\U0001FA00-\U0001FA6F"  # Chess Symbols
                               u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                               u"\U00002702-\U000027B0"  # Dingbats
                               u"\U000024C2-\U0001F251" 
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 멘션 제거
    mention_pattern = re.compile(r'@\S+')
    text = mention_pattern.sub(r'', text)
    
    # "@" 문자만 제거
    text = text.replace('@', '')
    
    # URL 제거
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    text = url_pattern.sub(r'', text)

    # HTML Entities 제거
    html_entity_pattern = re.compile(r'&[a-zA-Z]+;')
    text = html_entity_pattern.sub(r'', text)
    
    # 특수문자 제거
    special_char_pattern = re.compile(r'[\^_\+\/,\(\):*]')
    text = special_char_pattern.sub('', text)
    # 한글 제거
    korean_char_pattern = re.compile(r'[가-힣]+')
    text = korean_char_pattern.sub(r'', text)
    
    # "(cont.)" 제거
    cont_pattern = re.compile(r'\(cont\.\)\s*|\s*\(cont\.\)')
    text = cont_pattern.sub(r'', text)

    # 줄바꿈 제거
    text = text.replace('\n', ' ')

    # "fvck", "fuck", "porn", "pukimak", "sial", "sialan", "bajingan" 제거
    explicit_words_pattern = re.compile(r'fvck|fuck|porn|pukimak|sial|sialan|bajingan', flags=re.IGNORECASE)
    text = explicit_words_pattern.sub('', text)
    
    # "["와 "]" 특수 문자 제거
    text = text.replace('[', '').replace(']', '')

    # 연속된 "#" 기호를 공백으로 대체
    text = re.sub(r'#\s+#', ' ', text)
    
    return text



df['Text']=df['Text'].apply(clean_text)


# 'Text' 컬럼 중심으로 중복 제거
df.drop_duplicates(subset='Text', inplace=True)
df.reset_index(drop=True, inplace=True)

df.head(10)

<br>
<br>
<br>

# Tokenizing

In [ ]:
import spacy
import stanza
import pandas as pd
from tqdm import tqdm

# 스파이시 영어 모델 로드
nlp = spacy.load('en_core_web_sm')

# 스탄자 인도네시아어 모델 로드
stanza_id = stanza.Pipeline('id', processors='tokenize')

In [ ]:
def tokenizing(df):
    new_rows = []
    max_length = 5
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing rows"):
        article_text = row['Text']
        doc = nlp(article_text)
        sen_id = 1  # 문장 ID를 1로 초기화


        for sentence in doc.sents:  # spacy에서는 .sents로 문장에 접근
            stanza_doc = stanza_id(sentence.text)
            tokens = [token.text for sent in stanza_doc.sentences for token in sent.tokens]
            
            # 문장, 토큰, 문장길이, Sen_ID 컬럼 추가
            new_row = row.copy()
            new_row['Sentence'] = sentence.text
            new_row['Token'] = tokens
            new_row['Sen_len'] = len(tokens)
            new_row['Sen_ID'] = f"{row['Doc_ID']}_sen{sen_id:06d}"
            
            new_rows.append(new_row)
            sen_id += 1  # 문장 ID 증가


    # 리스트 기호 없이 단어 단위로 토큰화
    new_df = pd.DataFrame(new_rows)
    new_df['Pub_Type']='SNS'
    new_df['Tokenized_Sentence']= new_df['Token'].apply(' '.join)

    # 'Sentence' 컬럼 중심으로 중복 제거
    new_df.drop_duplicates(subset='Sentence', inplace=True)
    new_df.reset_index(drop=True, inplace=True)

    # 공식 제출용 컬럼명으로 컬럼명 변환
    new_df.rename(columns={'Sen_len': 'Word_Count'}, inplace=True)
    
    new_df = new_df[['Doc_ID', 'Filename', 'Title', 'Pub_Type', 'Pub_Subj', 'Pub_Date', 'Col_Date', 'Sen_ID', 'Word_Count', 'Text', 'Sentence','Tokenized_Sentence' , 'Token']]
    new_df = new_df[new_df['Token'].apply(lambda x: len(x) > max_length)]
    return new_df


In [ ]:
# 토큰화 완료된 최종 데이터프레임
df = tokenizing(df)
df.head(10)

In [ ]:
df.reset_index(drop=True, inplace=True)
df.sort_values(by='Sen_ID', inplace=True)
df